# 01 - Problem And Environment Walkthrough

Load `VectorConnectivityProblem` from the bundled `small_vector_001`
data, build a `VectorHabitatEnv` via `make_env`, reset it, and inspect
the v2 observation contract + action masks.

No training. No checkpoints. No deployment.

In [4]:
from pathlib import Path
pkg_root = Path.cwd().resolve()
if pkg_root.name != 'habconn':
    pkg_root = Path.cwd().parent
data_dir = pkg_root / 'data' / 'examples' / 'small_vector_001'
graphab_jar = pkg_root / 'tools' / 'graphab.jar'
work_root = pkg_root / 'tmp' / 'notebooks' / 'walkthrough'
work_root.mkdir(parents=True, exist_ok=True)
print(f'data_dir    : {data_dir}')
print(f'graphab_jar : {graphab_jar}')
print(f'work_root   : {work_root}')

data_dir    : C:\Users\dev\work\tum\drl-sp\08_pkg\habconn\data\examples\small_vector_001
graphab_jar : C:\Users\dev\work\tum\drl-sp\08_pkg\habconn\tools\graphab.jar
work_root   : C:\Users\dev\work\tum\drl-sp\08_pkg\habconn\tmp\notebooks\walkthrough


## Load the problem

In [5]:
from habconn.problems.vector_problem import VectorConnectivityProblem

problem = VectorConnectivityProblem.from_files(
    name='small_vector_001',
    vector_path=data_dir / 'candidates.shp',
    habitat_raster_path=data_dir / 'habitat.tif',
    resistance_raster_path=data_dir / 'resistance.tif',
    id_column='lyr_1',
    area_column='area',
    uniform_cost=1.0,
)
print(problem.summary())

{'name': 'small_vector_001', 'vector_path': 'C:\\Users\\dev\\work\\tum\\drl-sp\\08_pkg\\habconn\\data\\examples\\small_vector_001\\candidates.shp', 'habitat_raster_path': 'C:\\Users\\dev\\work\\tum\\drl-sp\\08_pkg\\habconn\\data\\examples\\small_vector_001\\habitat.tif', 'resistance_raster_path': 'C:\\Users\\dev\\work\\tum\\drl-sp\\08_pkg\\habconn\\data\\examples\\small_vector_001\\resistance.tif', 'n_planning_units': 79, 'id_column': 'lyr_1', 'internal_id_column': 'pu_id', 'cost_column': 'cost', 'eligibility_column': 'eligible', 'restored_resistance_value': 1.0, 'raster_width': 210, 'raster_height': 165, 'raster_crs': 'IGNF:ETRS89LAEA'}


## Build the environment

In [6]:
from habconn.training.make_env import make_env

env = make_env(
    data_dir=data_dir,
    graphab_jar=graphab_jar,
    work_root=work_root,
    budget=3,
    k=10,
    random_seed=42,
)
print('observation_space keys:', list(env.observation_space.spaces))
print('action_space          :', env.action_space)

observation_space keys: ['action_mask', 'budget_fraction', 'candidate_areas', 'candidate_costs', 'candidate_ids', 'current_pc', 'eligibility_mask', 'node_areas', 'node_costs', 'node_mask', 'remaining_budget', 'selected_fraction', 'selected_mask', 'step_count']
action_space          : Discrete(10)


## Reset and inspect the v2 observation

In [7]:
obs, info = env.reset(seed=42)
for key, arr in obs.items():
    print(f'{key:22s} shape={tuple(arr.shape)} dtype={arr.dtype}')

action_mask            shape=(10,) dtype=bool
candidate_ids          shape=(10,) dtype=int32
candidate_costs        shape=(10,) dtype=float32
candidate_areas        shape=(10,) dtype=float32
selected_mask          shape=(79,) dtype=bool
node_mask              shape=(79,) dtype=bool
node_costs             shape=(79,) dtype=float32
node_areas             shape=(79,) dtype=float32
eligibility_mask       shape=(79,) dtype=bool
remaining_budget       shape=(1,) dtype=float32
budget_fraction        shape=(1,) dtype=float32
step_count             shape=(1,) dtype=int32
selected_fraction      shape=(1,) dtype=float32
current_pc             shape=(1,) dtype=float32


## Action masks

MaskablePPO consumes `env.action_masks()` per step. Valid slots are
where the mask is True.

In [8]:
import numpy as np
masks = env.action_masks()
print('action_masks shape :', masks.shape)
print('valid count        :', int(masks.sum()))
print('valid slot indices :', np.flatnonzero(masks).tolist())

action_masks shape : (10,)
valid count        : 10
valid slot indices : [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]


## One environment step (no training)

Pick the first valid slot and step the env once. The reward magnitude
is raw delta-PC (~1e-6 in this bundled fixture).

In [9]:
action = int(np.flatnonzero(masks)[0])
obs, reward, terminated, truncated, info = env.step(action)
print(f'reward        : {reward:.6e}')
print(f'terminated    : {terminated}')
print(f'truncated     : {truncated}')
print(f'info keys     : {sorted(info)}')

reward        : 1.157504e-06
terminated    : False
truncated     : False
info keys     : ['action_type', 'backend_type', 'delta_pc', 'last_pu_id', 'n_feasible', 'pc_before', 'pc_value', 'remaining_budget', 'selected_pu_ids', 'step_count']
